## Launching model on dataset and collecting output

### Imports

In [ ]:
from datasets import load_dataset
from tqdm import tqdm

from langchain_core.prompts import ChatPromptTemplate
from services.common.llm_interface import LLMInterface
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
import os
import torch
import sys

sys.path.append("../../../services/")
from index import Index
from services.experiment.cropped.data_process_utils import retrieve_answer_token_index

torch.random.manual_seed(42)

/workspace/Diplom/.vsc_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '3')
print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}")


### Preparation

In [ ]:
ITERATIONS_COUNT = 12000

In [ ]:
torch.random.manual_seed(42)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(device)
print(device)
print(torch.cuda.get_device_properties(device))

cuda:0
_CudaDeviceProperties(name='NVIDIA L40', major=8, minor=9, total_memory=45385MB, multi_processor_count=142, uuid=441e6008-cd27-34b5-c81a-17dad9d4a894, L2_cache_size=96MB)


In [4]:
dataset = load_dataset("TIGER-Lab/MMLU-Pro", split="test")

In [ ]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Set HF_TOKEN environment variable before running this notebook")
login(token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN
)
tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
model = model.to(device)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 90.35it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_

In [7]:
chat_llm = LLMInterface(
    model=model,
    tokenizer=tokenizer,
    device=device
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an expert at answering multiple choice questions. 
            You will be given a question and options. Follow these rules strictly:
            
            1. Select the correct option number between 0 and 9
            2. Return your response as SINGLE token, only ONE number between 0 and 9

            Follow this output format:
            OPTION_NUMBER
            
            Example:
            2 

            Do not include any additional text except answer number between 0 and 9
            Answer the following question:
            """,
        ),
        (
            "human",
            """
            Question: {input_question}
    
            Options:
            {input_options}
            """,
        ),
    ]
)


llm_chain = prompt | chat_llm

In [8]:
len(dataset)

12032

### Running model on dataset and collecting results


In [9]:
def filter_func(model_response, right_answer):
    answer_token_index = retrieve_answer_token_index(model_response["score_data"])
    if answer_token_index is None or answer_token_index >= len(model_response["score_data"]) - 1:
        return False

    token_data = model_response["score_data"][answer_token_index]
    return right_answer in token_data["top_tokens"]

In [10]:
# launching model

ERROR_LIMIT = 500

index = Index(f"../../../index_data/llama3_MMLU-PRO_attn_cropped_{ITERATIONS_COUNT}")
index.clear()

correct_count = 0
progress_bar = tqdm(
    enumerate(dataset.select(range(ITERATIONS_COUNT))),
    total=ITERATIONS_COUNT,
    desc="Scoring MMLU-PRO",
)

for iter, data in progress_bar:
    for error_count in range(ERROR_LIMIT + 1):
        try:
            response = llm_chain.invoke(
                {
                    "input_question": dataset[iter]["question"],
                    "input_options": [
                        f"{i}:  {x}"
                        for i, x in enumerate(dataset[iter]["options"])
                    ]
                }
            )
            
            response["dataset_elem"] = dataset[iter]
        
            if filter_func(response, str(data["answer_index"])):
                if response["output_text"][-1] == str(
                    data["answer_index"]
                ):
                    correct_count += 1

                progress_bar.set_postfix(
                    {"Current accuracy": str(correct_count / (iter + 1))}
                )
                
                response = {
                    "iteration": iter,
                    **response
                }

                index.save_data(response, iter, logging=False)
                break
            
            if error_count == ERROR_LIMIT:
                raise Exception("Error limit exceeded")
            
        except Exception as e:
            if (error_count + 1) % 500 == 0:
                print(f"Error on iteration {iter + 1}\nError count: {error_count}\nMessage: {e}")
            if error_count == ERROR_LIMIT:
                print("Error limit exceeded")

File is deleted: ../../../index_data/llama3_MMLU-PRO_attn_cropped_12000_index.pkl
File is deleted: ../../../index_data/llama3_MMLU-PRO_attn_cropped_12000_data.pkl
Index is cleared successfully.


Scoring MMLU-PRO:   0%|          | 0/12000 [00:00<?, ?it/s]

Scoring MMLU-PRO:  42%|████▏     | 5079/12000 [27:31<303:23:55, 157.81s/it, Current accuracy=0.27761370348493797]

Error limit exceeded


Scoring MMLU-PRO: 100%|██████████| 12000/12000 [46:32<00:00,  4.30it/s, Current accuracy=0.24758333333333332]    


In [11]:
print(len(index))

11999
